In [0]:
import io
import requests
import zipfile
import pandas as pd
from pyspark.sql import functions as F

ESTBAN_URL = "https://www.bcb.gov.br/content/estatisticas/estatistica_bancaria_estban/municipio/{anoMes}_ESTBAN.csv.zip"
COOP_URL = "https://www.bcb.gov.br/content/estabilidadefinanceira/cosif/Cooperativas-de-credito/{anoMes}COOPERATIVAS.csv.zip"

def gerar_periodo(inicio, fim):
    return pd.date_range(start=pd.to_datetime(inicio, format="%Y%m"), 
                         end=pd.to_datetime(fim, format="%Y%m"), 
                         freq="MS").strftime("%Y%m").tolist()

def extrair_salvar_delta(url_base, nome_tabela, skiprows, data_inicio, data_fim):
    periodos = gerar_periodo(data_inicio, data_fim)
    dfs_pandas = []
    
    for ano_mes in periodos:
        url = url_base.format(anoMes=ano_mes)
        resp = requests.get(url)
        
        if resp.status_code == 200:
            with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
                with z.open(z.namelist()[0]) as f:
                    df_tmp = pd.read_csv(f, sep=";", encoding="latin1", engine="python", skiprows=skiprows, dtype=str)
                    df_tmp["ano_mes"] = str(ano_mes)
                    dfs_pandas.append(df_tmp)
                    
    if dfs_pandas:
        df_final_pd = pd.concat(dfs_pandas, ignore_index=True)
        
        # Sanitize column names for Delta Lake compatibility
        # Replace invalid characters and truncate to 255 chars (Unity Catalog limit)
        df_final_pd.columns = [
            col.replace('\t', '_').replace('+', '_').replace('/', '_').replace(' ', '_')[:255] 
            for col in df_final_pd.columns
        ]
        
        df_spark = spark.createDataFrame(df_final_pd)
        
        (df_spark.write
            .format("delta")
            .mode("overwrite") 
            .option("overwriteSchema", "true")
            .saveAsTable(f"bronze_{nome_tabela.lower()}"))
        
        print(f"Tabela bronze_{nome_tabela.lower()} salva com sucesso! Registros: {df_spark.count()}")

data_inicio, data_fim = "202412", "202603"
extrair_salvar_delta(ESTBAN_URL, "ESTBAN", 2, data_inicio, data_fim)
extrair_salvar_delta(COOP_URL, "COOP", 3, data_inicio, data_fim)